# Nesta semana estaremos fazendo um modelo de Recomendação para os nossos dadaos

In [1]:
# import zipfile
# import os
# import requests

# # Caminho do arquivo ZIP
# zip_url = "https://caelum-online-public.s3.amazonaws.com/challenge-spark/semanas-3-e-4.zip"
# zip_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Semanas_zip/semana-2.zip"
# extract_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Dataset"
         
# # Baixar o arquivo ZIP
# response = requests.get(zip_url)
# with open(zip_path, "wb") as f:
#     f.write(response.content)

# # Extrair o conteúdo
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extract_path)

# # Listar os arquivos extraídos
# os.listdir(extract_path)

In [2]:
from pyspark.sql import SparkSession

# Inicializar o Spark
spark = SparkSession.builder.appName("ProcessamentoParquet").getOrCreate()

# Caminho do arquivo PARQUET
parquet_path = "/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Dataset/dataset_ml_parquet"

# Carregar o arquivo JSON em um DataFrame
df = spark.read.parquet(parquet_path)

# Exibir as primeiras linhas
df.show(10)

your 131072x1 screen size is bogus. expect trouble
25/11/21 09:46:51 WARN Utils: Your hostname, Luiz resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/21 09:46:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/21 09:46:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|         bairro|condominio| iptu|    valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|
+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+
|00002dd9-cc74-480...|    2|       35|        1|      1|   0.0| 0.0|   Santo Cristo|     100.0|100.0| 245000.0|           1|

# Antes de começarmos nosso modelo, vamos tratá-los para funcionar em um modelo de machine learning do pyspark

In [3]:
X = df.columns
X.remove('id')
X.remove('bairro')

In [4]:
# Vetorizando dataset
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=X, outputCol='features')

df_vetorizado = assembler.transform(df)
df_vetorizado = df_vetorizado.select('features')
df_vetorizado.show(10, truncate=False)

25/11/21 09:46:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------------------------------------------------------------------------------+
|features                                                                                                    |
+------------------------------------------------------------------------------------------------------------+
|[2.0,35.0,1.0,1.0,0.0,0.0,100.0,100.0,245000.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]     |
|(23,[0,1,2,3,5,6,7,8,10,15,17,19,20,22],[1.0,84.0,2.0,2.0,1.0,770.0,105.0,474980.0,1.0,1.0,1.0,1.0,1.0,1.0])|
|(23,[1,2,3,6,7,8,12,14,17],[85.0,2.0,2.0,460.0,661.0,290000.0,1.0,1.0,1.0])                                 |
|(23,[1,2,3,5,6,7,8,11,18,19],[58.0,1.0,2.0,1.0,550.0,550.0,249000.0,1.0,1.0,1.0])                           |
|(23,[1,2,3,4,5,6,8,10],[64.0,2.0,2.0,1.0,1.0,850.0,530000.0,1.0])                                           |
|[0.0,200.0,6.0,4.0,4.0,2.0,2500.0,420.0,2900000.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0]  |
|

In [5]:
# Padronizando escalar dataset
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol='features', outputCol='scaler_features')

modelo_scaler = scaler.fit(df_vetorizado)
df_scaler = modelo_scaler.transform(df_vetorizado)
df_scaler.show(10,truncate=False)

+------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                    |scaler_features                                                                                                                                                                                                                                                                                                                                                                       |
+---------------------------

In [6]:
# Diminuindo a dimensionalidade dataset
from pyspark.ml.feature import PCA

pca = PCA(k=23, inputCol='scaler_features', outputCol='pca_features')

treino_pca = pca.fit(df_scaler)
treino_pca.explainedVariance

25/11/21 09:47:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/11/21 09:47:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
25/11/21 09:47:02 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


DenseVector([0.2655, 0.1721, 0.0913, 0.0544, 0.0522, 0.0466, 0.0443, 0.0416, 0.0347, 0.0272, 0.0244, 0.0201, 0.0192, 0.0176, 0.0155, 0.0139, 0.012, 0.0113, 0.0101, 0.0092, 0.0089, 0.0079, 0.0])

In [7]:
explica_array = treino_pca.explainedVariance.array

soma = 0
stop = 0

for i, valor in enumerate(explica_array):
    if(soma<=0.80):
        soma += explica_array[i]
    elif(stop==0):
        stop=1
        print(f"O íncie até 0.80 é {i-1}\nE seu valor é {soma - explica_array[i]}")

O íncie até 0.80 é 8
E seu valor é 0.7753463656098714


### Como da para ver, precisamos apenas de 8 variáveis para explicar 80% do nosso DataFrame

In [8]:
novo_pca = PCA(k=8, inputCol='scaler_features', outputCol='pca_features')

modelo_pca = novo_pca.fit(df_scaler)
df_pca = modelo_pca.transform(df_scaler)

df_final = df_pca.select('pca_features')
df_final = df_final.withColumnRenamed('pca_features', 'features')
df_final.show(10, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                             |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[-6.165125049533942,1.338098526940416,-1.7052299823815522,-0.5338289630545653,0.08903815478586079,-0.3134396180249441,-5.880683742692612,-4.3463129306499475]        |
|[-3.252911181218317,-1.1179591836230234,-0.2923895841499696,3.1955388200208477,0.15286306982185222,1.3028930490659623,-0.43693999810125045,0.05205299467369569]      |
|[-1.061176932962965,-1.6685040058693055,-2.3075948278392087,0.10553124125771536,-0.06914386452865218,0.7187181899564261,0.08236222790223582,0.14780594823036006

# Feito o tratamento dos dados, utilizaremos o model Kmeans para a realização do nosso modelo de recomendação.

In [9]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(featuresCol='features', metricName='silhouette', distanceMeasure='squaredEuclidean')

def calcula_cluster(df, numero_clusters):
    kmeans = KMeans(featuresCol='features', k=numero_clusters, seed=390)
    modelo = kmeans.fit(df)
    inercia = modelo.summary.trainingCost
    silhueta = evaluator.evaluate(modelo.transform(df))
    print("-"*15)
    print(f"Clusteres: {numero_clusters}")
    print(f"Inérica: {inercia}")
    print(f"Silhueta: {silhueta}")

In [10]:
for i in range(2,21):
    calcula_cluster(df_final, i)

---------------
Clusteres: 2
Inérica: 847452.0401333753
Silhueta: 0.4535923421955306
---------------
Clusteres: 3
Inérica: 724760.16004923
Silhueta: 0.45442321564956567
---------------
Clusteres: 4
Inérica: 655218.172007023
Silhueta: 0.380591156266863
---------------
Clusteres: 5
Inérica: 621093.5800852601
Silhueta: 0.26214014836694843
---------------
Clusteres: 6
Inérica: 558005.7374814001
Silhueta: 0.353455452820209
---------------
Clusteres: 7
Inérica: 523146.1774836836
Silhueta: 0.382916584879627
---------------
Clusteres: 8
Inérica: 495961.9712743556
Silhueta: 0.3471956702531843
---------------
Clusteres: 9
Inérica: 431236.5867810906
Silhueta: 0.40815164806943877
---------------
Clusteres: 10
Inérica: 432156.5825380186
Silhueta: 0.35192490732369514
---------------
Clusteres: 11
Inérica: 351964.6823473248
Silhueta: 0.465005188634595
---------------
Clusteres: 12
Inérica: 315833.7787650272
Silhueta: 0.513187031218173
---------------
Clusteres: 13
Inérica: 311034.3316676787
Silhueta:

In [11]:
melhores_cluesteres = [12, 13, 14, 15, 17, 18, 19]

In [12]:
def ordena_cluster(df, numero_clusters):
    kmeans = KMeans(featuresCol='features', k=numero_clusters, seed=390)
    modelo = kmeans.fit(df)
    df_clusterizado = modelo.transform(df)
    print(df_clusterizado.groupBy('prediction').count().orderBy('count', ascending=False).show())

In [13]:
for i, valor in enumerate(melhores_cluesteres):
    print('-'*25)
    print(f'Quantidade: {melhores_cluesteres[i]}')
    ordena_cluster(df_final, valor)

-------------------------
Quantidade: 12
+----------+-----+
|prediction|count|
+----------+-----+
|         5|12921|
|        10| 9945|
|         3| 7849|
|         6| 6811|
|         0| 6472|
|         2| 5840|
|        11| 4560|
|         1| 3818|
|         4| 3748|
|         7| 3416|
|         9| 1144|
|         8|   27|
+----------+-----+

None
-------------------------
Quantidade: 13
+----------+-----+
|prediction|count|
+----------+-----+
|         2|14629|
|        10|10011|
|         3| 8006|
|         1| 7026|
|         6| 6493|
|         0| 4909|
|         7| 4309|
|        11| 3874|
|         5| 3178|
|        12| 2869|
|         9| 1144|
|         8|   90|
|         4|   13|
+----------+-----+

None
-------------------------
Quantidade: 14
+----------+-----+
|prediction|count|
+----------+-----+
|         0|10972|
|         6| 9624|
|        12| 7506|
|         1| 6756|
|        10| 5867|
|         2| 5745|
|         4| 4617|
|        11| 4556|
|         5| 4198|
|         

### De acordo com nossas análises o modelo com 17 clusteres apresentou o melhor resultado, pois seu erro (inércia), foi de 226 mil, um ótimo valor para casas com precificação de 150 mil à mais de 1 milhão, além de que sua silhueta foi uma dos que mais ficou perto de 1, ou seja, os dados estão bem agrupados em torno do ponto central do cluester. 

### Por fim analisando pela quantidade de casas que foram agrupadas temos que está bem a distribuição entre elas, o que tudo nos aponta para utilizar este modelo.

### Vamos crirar um Pipeline para tratar os dados e fazer um filtro e extrar as informações de ID de um imóvel e verificar quais imóveis pertencem ao mesmo cluster.

In [14]:
from pyspark.ml.pipeline import Pipeline

kmeans = KMeans(featuresCol='features', k=17, seed=390)

pipeline = Pipeline(stages=[assembler, modelo_scaler, modelo_pca, kmeans])

In [15]:
modelo_final = pipeline.fit(df)

df_final_clusterizado = modelo_final.transform(df)

df_final_clusterizado.show(5)

+--------------------+-----+---------+---------+-------+------+----+------------+----------+-----+--------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+--------------------+--------------------+----------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|      bairro|condominio| iptu|   valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|            features|     scaler_features|        pca_features|prediction|
+--------------------+-----+---------+---------+-------+------+----+------------+----------+-----+--------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+-----

In [ ]:
def cluster_imovel(df, numero_cluster):
    
    return resultado

In [17]:
busca = cluster_imovel(df_final_clusterizado, 10)
busca.show(truncate=False)

+------------------------------------+-----+---------+---------+-------+------+----+---------------+----------+-------+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+-------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|id                  

# Criaremos por fim uma função recomendadora baseada nos 10 imóveis mais próximos cluster a cluster

In [176]:
def vetores_cluster(df, numero_cluster):
    cluster = df.filter(df['prediction'] == numero_cluster)

    partes_cluster = cluster.collect()
    vetores_cluster = []

    for i in range(cluster.count()):
        partes = partes_cluster[i]
        id_cluster = [parte for parte in partes][0]
        feature_cluster = [parte for parte in partes][-2]

        vetores_cluster.append((id_cluster, feature_cluster))


    return vetores_cluster

In [177]:
clusteres = []

for i in range(1, 18):
    clusteres.append(vetores_cluster(df_final_clusterizado, i))

In [163]:
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.functions import udf
import numpy as np

@udf(returnType=ArrayType(StringType()))
def calcula_10_proximos(id_imovel, vetor_imovel, cluster_imovel):
    cluster_utilizado = clusteres[cluster_imovel]

    distancias = []

    for i in range(len(cluster_utilizado)):
        id, vetor = cluster_utilizado[i]

        if id != id_imovel:
           distancias.append((id, np.linalg.norm(vetor - vetor_imovel)))

    distancias = sorted(distancias, key=lambda x: x[1])
    
    distancias_10 = []
    for i in range(10):
        ids, valor = distancias[i]
        distancias_10.append(ids) 

    return distancias_10

In [173]:
from pyspark.sql.functions import col

df_finalizado = df_final_clusterizado.select('*')

df_finalizado = df_finalizado.withColumn('10 Próximos', calcula_10_proximos(col('id'), col('pca_features'), col('prediction')))
df_finalizado = df_finalizado.drop(*['features', 'scaler_features', 'pca_features', 'prediction'])

df_finalizado.show(10)

+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+
|                  id|andar|area_util|banheiros|quartos|suites|vaga|         bairro|condominio| iptu|    valor|Zona Central|Zona Norte|Zona Oeste|Zona Sul|Academia|Animais permitidos|Churrasqueira|Condomínio fechado|Elevador|Piscina|Playground|Portaria 24h|Portão eletrônico|Salão de festas|         10 Próximos|
+--------------------+-----+---------+---------+-------+------+----+---------------+----------+-----+---------+------------+----------+----------+--------+--------+------------------+-------------+------------------+--------+-------+----------+------------+-----------------+---------------+--------------------+
|00002dd9-cc74-480...|    2|       35|        1|      1|   0.

### Com isto, finalizamos a semana fazendo nosso pré-modelo de recomendação, colocaremos tudo agora em um arquivo python para finalizar e prever para quantas imóveis próximos quisermos